27 — F3 Canonical Target + Environment Contract
Bu notebook V2 denetimindeki F3 bulgusunu kapatır.

Kanonik eğitim sözleşmesi:

X = final_online_training_features.npz::X_features → float32
y = final_online_training_manifest.csv::soh_rated_pct → float64
battery_id, cycle_uid → manifest
RF20x7: n_estimators=20, max_depth=7, max_features="sqrt", min_samples_leaf=2, random_state=42, median imputer
doğrulanmış scikit-learn = 1.6.1
Mevcut doğru fitted model değiştirilmez. Ama tekrar üretim için float64 target içeren yeni bir canonical_v2.npz, environment kaydı ve eğitim sözleşmesi oluşturulur.

In [1]:

from pathlib import Path
from datetime import datetime
import hashlib, json, platform, sys, importlib.metadata as ilm

import numpy as np
import pandas as pd
import joblib
import sklearn
from sklearn.base import clone

ROOT = Path(r"${PROJECT_ROOT}")
RESULTS = ROOT / "results"
DEPLOY = ROOT / "deployment" / "final_online_rf20x7"

MANIFEST = DEPLOY / "final_online_training_manifest.csv"
FEATURES = DEPLOY / "final_online_training_features.npz"
PIPELINE = DEPLOY / "final_online_rf20x7_pipeline.pkl"
HASHES = DEPLOY / "final_online_artifact_hashes.csv"
SEED_BATTERY = RESULTS / "17_error_uncertainty_reproducibility_freeze" / "seed_stability_by_battery.csv"

OUT = RESULTS / "27_F3_canonical_target_environment_contract"
OUT.mkdir(parents=True, exist_ok=True)

CANONICAL_NPZ = OUT / "final_online_training_canonical_v2.npz"

EXPECTED_SKLEARN = "1.6.1"
SEED = 42

for p in [MANIFEST, FEATURES, PIPELINE, HASHES, SEED_BATTERY]:
    if not p.exists():
        raise FileNotFoundError(p)

print("Python:", sys.version)
print("scikit-learn:", sklearn.__version__)

if sklearn.__version__ != EXPECTED_SKLEARN:
    raise RuntimeError(
        f"F3 STOP: use scikit-learn {EXPECTED_SKLEARN}; current={sklearn.__version__}"
    )


Python: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]
scikit-learn: 1.6.1


In [2]:

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            b = f.read(1024*1024)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

hdf = pd.read_csv(HASHES)
expected = {str(r["artifact"]): str(r["sha256"]) for _, r in hdf.iterrows()}

rows = []
for p in [MANIFEST, FEATURES, PIPELINE]:
    actual = sha256_file(p)
    exp = expected.get(p.name)
    rows.append({
        "artifact": p.name,
        "expected_sha256": exp,
        "actual_sha256": actual,
        "pass": exp is not None and exp == actual,
    })

source_hash_check = pd.DataFrame(rows)
display(source_hash_check)

if not source_hash_check["pass"].all():
    raise RuntimeError("D1 source hash mismatch.")

print("SOURCE HASH CHECK: PASS")


,artifact,expected_sha256,actual_sha256,pass
0,final_online_training_manifest.csv,f177295b5c20b9ec1246e5f40d3652f0eb86bc42bbae16...,f177295b5c20b9ec1246e5f40d3652f0eb86bc42bbae16...,True
1,final_online_training_features.npz,1faf11802ab082e470b26257c8e34a9eb33c50a70ebea8...,1faf11802ab082e470b26257c8e34a9eb33c50a70ebea8...,True
2,final_online_rf20x7_pipeline.pkl,2ad2b2ca89ab0a5aa09cffe3df663bbbb49d3e198903e3...,2ad2b2ca89ab0a5aa09cffe3df663bbbb49d3e198903e3...,True


SOURCE HASH CHECK: PASS


In [3]:

manifest = pd.read_csv(MANIFEST)
manifest["battery_id"] = manifest["battery_id"].astype(str)
manifest["cycle_uid"] = manifest["cycle_uid"].astype(str)

npz = np.load(FEATURES, allow_pickle=False)
X = np.asarray(npz["X_features"], dtype=np.float32)
y_npz32 = np.asarray(npz["y"], dtype=np.float32)
y64 = manifest["soh_rated_pct"].to_numpy(dtype=np.float64)

battery_id = manifest["battery_id"].to_numpy(dtype=str)
cycle_uid = manifest["cycle_uid"].to_numpy(dtype=str)

if X.shape != (2521, 44):
    raise RuntimeError(X.shape)

diff = y64 - y_npz32.astype(np.float64)

target_summary = pd.DataFrame([{
    "rows": len(X),
    "features": X.shape[1],
    "X_dtype": str(X.dtype),
    "legacy_npz_y_dtype": str(y_npz32.dtype),
    "canonical_y_dtype": str(y64.dtype),
    "max_abs_target_diff_pp": float(np.max(np.abs(diff))),
    "mean_abs_target_diff_pp": float(np.mean(np.abs(diff))),
    "nonzero_rows": int(np.sum(diff != 0)),
}])

display(target_summary)

np.savez_compressed(
    CANONICAL_NPZ,
    X_features=X,
    y_soh_rated_pct=y64,
    battery_id=battery_id,
    cycle_uid=cycle_uid,
)

rt = np.load(CANONICAL_NPZ, allow_pickle=False)

roundtrip_pass = (
    np.array_equal(rt["X_features"], X)
    and np.array_equal(rt["y_soh_rated_pct"], y64)
    and np.array_equal(rt["battery_id"].astype(str), battery_id)
    and np.array_equal(rt["cycle_uid"].astype(str), cycle_uid)
)

print("Canonical NPZ round-trip:", roundtrip_pass)
print("Canonical NPZ SHA256:", sha256_file(CANONICAL_NPZ))

if not roundtrip_pass:
    raise RuntimeError("Canonical NPZ round-trip failed.")


,rows,features,X_dtype,legacy_npz_y_dtype,canonical_y_dtype,max_abs_target_diff_pp,mean_abs_target_diff_pp,nonzero_rows
0,2521,44,float32,float32,float64,0.000004,0.000002,2521


Canonical NPZ round-trip: True
Canonical NPZ SHA256: 4472effc044206968c759c3f133d8216efb6dec7abae02074972f15951318962


In [4]:

saved = joblib.load(PIPELINE)
params = saved.get_params()

expected_params = {
    "model__n_estimators": 20,
    "model__max_depth": 7,
    "model__max_features": "sqrt",
    "model__min_samples_leaf": 2,
    "model__random_state": 42,
}

param_rows = []
for k, exp in expected_params.items():
    obs = params.get(k)
    param_rows.append({"parameter": k, "expected": exp, "observed": obs, "pass": obs == exp})

param_check = pd.DataFrame(param_rows)
display(param_check)

if not param_check["pass"].all():
    raise RuntimeError("RF20x7 parameter mismatch.")

recon = clone(saved)
if "model__n_jobs" in recon.get_params():
    recon.set_params(model__n_jobs=1)

recon.fit(X, y64)

saved_model = saved.named_steps["model"]
recon_model = recon.named_steps["model"]

tree_rows = []
for i, (a, b) in enumerate(zip(saved_model.estimators_, recon_model.estimators_)):
    ta, tb = a.tree_, b.tree_
    exact = (
        ta.node_count == tb.node_count
        and np.array_equal(ta.children_left, tb.children_left)
        and np.array_equal(ta.children_right, tb.children_right)
        and np.array_equal(ta.feature, tb.feature)
        and np.array_equal(ta.threshold, tb.threshold)
        and np.array_equal(ta.value, tb.value)
    )
    tree_rows.append({
        "tree_index": i,
        "saved_nodes": ta.node_count,
        "reconstructed_nodes": tb.node_count,
        "exact_tree_match": exact,
    })

tree_compare = pd.DataFrame(tree_rows)
display(tree_compare)

exact_tree_count = int(tree_compare["exact_tree_match"].sum())
print("Canonical float64 exact tree match:", exact_tree_count, "/20")

if exact_tree_count != 20:
    raise RuntimeError("Canonical float64 y did not reproduce 20/20 final trees.")


,parameter,expected,observed,pass
0,model__n_estimators,20,20,True
1,model__max_depth,7,7,True
2,model__max_features,sqrt,sqrt,True
3,model__min_samples_leaf,2,2,True
4,model__random_state,42,42,True


,tree_index,saved_nodes,reconstructed_nodes,exact_tree_match
0,0,175,175,True
1,1,197,197,True
2,2,203,203,True
3,3,195,195,True
4,4,215,215,True
5,5,187,187,True
6,6,191,191,True
7,7,221,221,True
8,8,203,203,True
9,9,211,211,True


Canonical float64 exact tree match: 20 /20


In [5]:

frozen = pd.read_csv(SEED_BATTERY)
frozen["test_battery"] = frozen["test_battery"].astype(str)

frozen20 = frozen[
    (frozen["model"] == "RF_20x7_arch")
    & (pd.to_numeric(frozen["seed"]) == SEED)
][["test_battery", "mae_pp"]].copy()

if len(frozen20) != 34:
    raise RuntimeError(f"Expected 34 frozen rows; got {len(frozen20)}")

barray = manifest["battery_id"].to_numpy(dtype=str)
lobo_rows = []

for test_battery in sorted(manifest["battery_id"].unique()):
    te = barray == test_battery
    tr = ~te

    m = clone(saved)
    if "model__n_jobs" in m.get_params():
        m.set_params(model__n_jobs=1)

    m.fit(X[tr], y64[tr])
    pred = m.predict(X[te])
    err = pred - y64[te]

    lobo_rows.append({
        "test_battery": test_battery,
        "test_cycles": int(te.sum()),
        "mae_pp": float(np.mean(np.abs(err))),
        "rmse_pp": float(np.sqrt(np.mean(err**2))),
        "bias_pp": float(np.mean(err)),
    })

canonical_lobo = pd.DataFrame(lobo_rows)

cmp = canonical_lobo.merge(
    frozen20.rename(columns={"mae_pp": "frozen_mae_pp"}),
    on="test_battery",
    how="left",
    validate="one_to_one",
)

cmp["abs_mae_diff_pp"] = np.abs(cmp["mae_pp"] - cmp["frozen_mae_pp"])

macro_mae = float(canonical_lobo["mae_pp"].mean())
max_battery_diff = float(cmp["abs_mae_diff_pp"].max())

display(cmp)

print("Canonical RF20x7 macro MAE pp:", macro_mae)
print("Max per-battery MAE diff vs frozen pp:", max_battery_diff)

if not np.isclose(macro_mae, 7.096415349426006, atol=1e-10):
    raise RuntimeError("Canonical LOBO macro mismatch.")

if max_battery_diff > 1e-10:
    raise RuntimeError("Canonical per-battery LOBO mismatch.")

print("CANONICAL 34-BATTERY LOBO: PASS")


,test_battery,test_cycles,mae_pp,rmse_pp,bias_pp,frozen_mae_pp,abs_mae_diff_pp
0,B0005,168,4.939601,6.327978,4.164123,4.939601,1.776357e-15
1,B0006,168,7.511388,8.677186,-0.639164,7.511388,2.664535e-15
2,B0007,168,3.623494,4.305942,-0.325946,3.623494,4.884981e-15
3,B0018,132,6.775982,7.308509,6.611175,6.775982,7.993606e-15
4,B0025,28,2.215253,2.824648,-2.215253,2.215253,1.154632e-14
5,B0026,27,0.872183,1.592378,0.872163,0.872183,5.773160e-15
6,B0027,28,12.266581,12.770019,-12.266581,12.266581,1.598721e-14
7,B0028,28,11.261488,11.531354,-11.261488,11.261488,2.309264e-14
8,B0029,40,3.765293,4.546395,-3.419217,3.765293,5.773160e-15
9,B0030,40,3.978839,4.590839,3.687368,3.978839,4.440892e-15


Canonical RF20x7 macro MAE pp: 7.096415349426006
Max per-battery MAE diff vs frozen pp: 2.3092638912203256e-14
CANONICAL 34-BATTERY LOBO: PASS


In [6]:

legacy = clone(saved)
if "model__n_jobs" in legacy.get_params():
    legacy.set_params(model__n_jobs=1)

legacy.fit(X, y_npz32)
legacy_model = legacy.named_steps["model"]

legacy_rows = []
for i, (a, b) in enumerate(zip(saved_model.estimators_, legacy_model.estimators_)):
    ta, tb = a.tree_, b.tree_
    exact = (
        ta.node_count == tb.node_count
        and np.array_equal(ta.children_left, tb.children_left)
        and np.array_equal(ta.children_right, tb.children_right)
        and np.array_equal(ta.feature, tb.feature)
        and np.array_equal(ta.threshold, tb.threshold)
        and np.array_equal(ta.value, tb.value)
    )
    legacy_rows.append({"tree_index": i, "exact_tree_match": exact})

legacy_tree_compare = pd.DataFrame(legacy_rows)
legacy_exact_count = int(legacy_tree_compare["exact_tree_match"].sum())

display(legacy_tree_compare)
print("Legacy NPZ float32 y exact tree matches:", legacy_exact_count, "/20")


,tree_index,exact_tree_match
0,0,False
1,1,False
2,2,False
3,3,False
4,4,False
5,5,False
6,6,False
7,7,False
8,8,False
9,9,False


Legacy NPZ float32 y exact tree matches: 0 /20


In [7]:
packages = ["numpy", "pandas", "scikit-learn", "joblib", "scipy", "matplotlib", "nbformat"]
versions = {}

for pkg in packages:
    try:
        versions[pkg] = ilm.version(pkg)
    except ilm.PackageNotFoundError:
        versions[pkg] = None

env = {
    "contract_id": "CANONICAL_ONLINE_RF20X7_TRAINING_CONTRACT_V1",
    "protocol_id": "ONLINE_FIXED_START_CAUSAL_10S_V1",
    "created_at": datetime.now().astimezone().isoformat(),
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "packages": versions,
    "canonical_X": "final_online_training_features.npz::X_features",
    "canonical_X_dtype": "float32",
    "canonical_y": "final_online_training_manifest.csv::soh_rated_pct",
    "canonical_y_dtype": "float64",
    "canonical_npz": CANONICAL_NPZ.name,
    "canonical_npz_sha256": sha256_file(CANONICAL_NPZ),
    "source_manifest_sha256": sha256_file(MANIFEST),
    "source_features_sha256": sha256_file(FEATURES),
    "source_pipeline_sha256": sha256_file(PIPELINE),
    "exact_tree_matches": exact_tree_count,
    "legacy_float32_tree_matches": legacy_exact_count,
    "canonical_lobo_macro_mae_pp": macro_mae,
    "max_battery_mae_diff_vs_frozen_pp": max_battery_diff,
}

with (OUT / "canonical_training_environment.json").open("w", encoding="utf-8") as f:
    json.dump(env, f, indent=2, ensure_ascii=False)

req_lines = []
for pkg in packages:
    if versions[pkg] is not None:
        req_lines.append(f"{pkg}=={versions[pkg]}")

(OUT / "requirements_repro_exact.txt").write_text(
    "\n".join(req_lines) + "\n",
    encoding="utf-8",
)

yaml = [
    "name: nasa-soh-repro",
    "channels:",
    "  - conda-forge",
    "dependencies:",
    f"  - python={platform.python_version()}",
    "  - pip",
    "  - pip:",
]
yaml += [f"      - {x}" for x in req_lines]

(OUT / "environment_repro_exact.yml").write_text(
    "\n".join(yaml) + "\n",
    encoding="utf-8",
)

contract = f"""NASA SOH — CANONICAL TRAINING CONTRACT

Protocol:
ONLINE_FIXED_START_CAUSAL_10S_V1

Canonical X:
{FEATURES}
array=X_features
dtype=float32

Canonical y:
{MANIFEST}
column=soh_rated_pct
dtype=float64

Do NOT use the legacy NPZ float32 y array as the canonical retraining target.

Canonical portable combined artifact:
{CANONICAL_NPZ}
SHA256:
{sha256_file(CANONICAL_NPZ)}

Final fitted pipeline:
{PIPELINE}
SHA256:
{sha256_file(PIPELINE)}

Verified scikit-learn:
{EXPECTED_SKLEARN}

Verification:
canonical target tree matches = {exact_tree_count}/20
legacy float32 target tree matches = {legacy_exact_count}/20
canonical LOBO macro MAE = {macro_mae:.12f} pp
max battery MAE diff vs frozen = {max_battery_diff:.3e} pp
"""

(OUT / "README_CANONICAL_TRAINING_CONTRACT.txt").write_text(
    contract,
    encoding="utf-8",
)

print(json.dumps(env, indent=2, ensure_ascii=False))


{
  "contract_id": "CANONICAL_ONLINE_RF20X7_TRAINING_CONTRACT_V1",
  "protocol_id": "ONLINE_FIXED_START_CAUSAL_10S_V1",
  "created_at": "2026-10-01T14:24:48.416958+03:00",
  "python_version": "3.13.5",
  "platform": "Windows-11-10.0.26300-SP0",
  "packages": {
    "numpy": "2.1.3",
    "pandas": "2.2.3",
    "scikit-learn": "1.6.1",
    "joblib": "1.4.2",
    "scipy": "1.15.3",
    "matplotlib": "3.10.0",
    "nbformat": "5.10.4"
  },
  "canonical_X": "final_online_training_features.npz::X_features",
  "canonical_X_dtype": "float32",
  "canonical_y": "final_online_training_manifest.csv::soh_rated_pct",
  "canonical_y_dtype": "float64",
  "canonical_npz": "final_online_training_canonical_v2.npz",
  "canonical_npz_sha256": "4472effc044206968c759c3f133d8216efb6dec7abae02074972f15951318962",
  "source_manifest_sha256": "f177295b5c20b9ec1246e5f40d3652f0eb86bc42bbae169d03781758e2e74f40",
  "source_features_sha256": "1faf11802ab082e470b26257c8e34a9eb33c50a70ebea8ff385c4043e2f30acd",
  "source

In [8]:

target_summary.to_csv(OUT / "target_precision_audit.csv", index=False, encoding="utf-8-sig")
param_check.to_csv(OUT / "rf20x7_parameter_check.csv", index=False, encoding="utf-8-sig")
tree_compare.to_csv(OUT / "canonical_float64_tree_exact_match.csv", index=False, encoding="utf-8-sig")
legacy_tree_compare.to_csv(OUT / "legacy_float32_target_tree_match.csv", index=False, encoding="utf-8-sig")
canonical_lobo.to_csv(OUT / "canonical_float64_seed42_lobo_by_battery.csv", index=False, encoding="utf-8-sig")
cmp.to_csv(OUT / "canonical_lobo_vs_frozen_comparison.csv", index=False, encoding="utf-8-sig")

artifact_rows = []
for p in sorted(OUT.glob("*")):
    if p.is_file() and p.name != "F3_artifact_manifest_sha256.csv":
        artifact_rows.append({
            "filename": p.name,
            "size_bytes": p.stat().st_size,
            "sha256": sha256_file(p),
        })

artifact_manifest = pd.DataFrame(artifact_rows)
artifact_manifest.to_csv(
    OUT / "F3_artifact_manifest_sha256.csv",
    index=False,
    encoding="utf-8-sig",
)

display(artifact_manifest)


,filename,size_bytes,sha256
0,canonical_float64_seed42_lobo_by_battery.csv,2293,8beb460700a6652d7099dba8af47182b824797393635c5...
1,canonical_float64_tree_exact_match.csv,394,03ac21b221db73b07d5c0e1579357c12052b335e67ae5a...
2,canonical_lobo_vs_frozen_comparison.csv,3635,c70c5a07985a2f05060e47c33c62cf374c6243459241db...
3,canonical_training_environment.json,1294,cffbb78d32feea7fa00ece67b35f48a96a9011787c8ba4...
4,environment_repro_exact.yml,277,f8cd7bc3ae454068a1d0278cc16e0d0aaa4a5cd8f00658...
5,final_online_training_canonical_v2.npz,353429,4472effc044206968c759c3f133d8216efb6dec7abae02...
6,legacy_float32_target_tree_match.csv,222,e36e7f5e7658faf942e46503cf8d81ce191b10044710f9...
7,README_CANONICAL_TRAINING_CONTRACT.txt,1237,30c45f9be77fa4db71f5717cc7428e9db455b327967041...
8,requirements_repro_exact.txt,118,f33ba8c4f2871857cf7393f97e844929caa529150f679c...
9,rf20x7_parameter_check.csv,198,b46e4f86658d6a255cf4a3a5876863251653a6fc99e54b...


In [9]:

f3_checks = pd.DataFrame([
    {"check": "scikit-learn==1.6.1", "pass": sklearn.__version__ == EXPECTED_SKLEARN},
    {"check": "source hashes verified", "pass": bool(source_hash_check["pass"].all())},
    {"check": "canonical X float32", "pass": str(X.dtype) == "float32"},
    {"check": "canonical y float64", "pass": str(y64.dtype) == "float64"},
    {"check": "canonical NPZ round-trip exact", "pass": bool(roundtrip_pass)},
    {"check": "RF20x7 parameters verified", "pass": bool(param_check["pass"].all())},
    {"check": "canonical float64 reconstructs 20/20 trees", "pass": exact_tree_count == 20},
    {"check": "34-battery LOBO matches frozen battery MAEs", "pass": max_battery_diff <= 1e-10},
])

display(f3_checks)

f3_pass = bool(f3_checks["pass"].all())

print("=" * 100)
print("F3 CANONICAL TARGET + ENVIRONMENT CONTRACT — FINAL SUMMARY")
print("=" * 100)

print("Canonical X dtype:", X.dtype)
print("Canonical y dtype:", y64.dtype)
print("Max NPZ-float32 vs manifest-float64 target diff pp:", float(np.max(np.abs(diff))))

print()
print("FINAL MODEL RECONSTRUCTION")
print("Exact canonical float64 tree matches:", exact_tree_count, "/20")
print("Exact legacy NPZ float32 tree matches:", legacy_exact_count, "/20")

print()
print("LOBO")
print("Canonical 34-battery RF20x7 macro MAE pp:", macro_mae)
print("Max per-battery MAE diff vs frozen pp:", max_battery_diff)

print()
print("ENVIRONMENT")
print("Python:", platform.python_version())
print("scikit-learn:", sklearn.__version__)
print("NumPy:", versions["numpy"])
print("pandas:", versions["pandas"])
print("joblib:", versions["joblib"])

print()
print("CANONICAL ARTIFACT")
print(CANONICAL_NPZ)
print("SHA256:", sha256_file(CANONICAL_NPZ))

print()
print("F3 STATUS:", "PASS" if f3_pass else "FAIL")

if not f3_pass:
    raise RuntimeError("F3 failed.")


,check,pass
0,scikit-learn==1.6.1,True
1,source hashes verified,True
2,canonical X float32,True
3,canonical y float64,True
4,canonical NPZ round-trip exact,True
5,RF20x7 parameters verified,True
6,canonical float64 reconstructs 20/20 trees,True
7,34-battery LOBO matches frozen battery MAEs,True


F3 CANONICAL TARGET + ENVIRONMENT CONTRACT — FINAL SUMMARY
Canonical X dtype: float32
Canonical y dtype: float64
Max NPZ-float32 vs manifest-float64 target diff pp: 3.8132664172962905e-06

FINAL MODEL RECONSTRUCTION
Exact canonical float64 tree matches: 20 /20
Exact legacy NPZ float32 tree matches: 0 /20

LOBO
Canonical 34-battery RF20x7 macro MAE pp: 7.096415349426006
Max per-battery MAE diff vs frozen pp: 2.3092638912203256e-14

ENVIRONMENT
Python: 3.13.5
scikit-learn: 1.6.1
NumPy: 2.1.3
pandas: 2.2.3
joblib: 1.4.2

CANONICAL ARTIFACT
${PROJECT_ROOT}\results\27_F3_canonical_target_environment_contract\final_online_training_canonical_v2.npz
SHA256: 4472effc044206968c759c3f133d8216efb6dec7abae02074972f15951318962

F3 STATUS: PASS
